### Exploratory Data Analysis
The first step of this project involves thoroughly understanding the dataset to evaluate how the data is structured and stored within the database. This includes examining data distribution, relationships between tables, key variables, and overall data quality.

The objective of this exploratory analysis is to identify whether additional aggregated metrics or derived features need to be created to support business decision-making.

Specifically, the analysis aims to generate insights that can assist in:

* Vendor Selection for Profitability
    Identifying high-performing vendors based on revenue, margins, cost efficiency, and sales volume to support strategic sourcing decisions.

* Product Pricing Optimization
    Analyzing pricing patterns, cost structures, and demand behavior to determine optimal pricing strategies that maximize revenue and profit margins.

In [2]:
import pandas as pd
import sqlite3
import time 

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)


In [4]:
# creating database connection 
conn = sqlite3.connect('inventory.db')

In [5]:
# Checking the tables present in the databse
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'",conn)
tables

,name


In [11]:
for table in tables['name']:
    print('-'*50,f'{table}','-'*50)
    print(pd.read_sql(f'select count(*) as count from {table}',conn).squeeze())
    display(pd.read_sql(f'select * from {table} limit 5' ,conn))

 * print(pd.read_sql(f 'select count(*) as count from {table}',conn)['count'].value[0]) -- same as below 
 * print(pd.read_sql(f 'select count(*) as count from {table}',conn)['count'].iloc[0])  --   better way
 * profesion way .squeeze() converts single-value DataFrame → scalar automatically

In [10]:
vendor_num = "4466"

In [12]:
purchases = pd.read_sql(f'SELECT * FROM purchases where vendorNumber = {vendor_num}',conn)
purchases

DatabaseError: Execution failed on sql 'SELECT * FROM purchases where vendorNumber = 4466': no such table: purchases

In [9]:
purchase_prices = pd.read_sql(f"""SELECT * FROM purchase_prices WHERE VendorNumber = {vendor_num}""",conn)
purchase_prices

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.41,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.35,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


In [10]:
sales = pd.read_sql(f'SELECT * from sales where VendorNo = {vendor_num}',conn)
sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9448,9_BLACKPOOL_5215,9,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-12-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9449,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-02,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9450,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9451,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


In [11]:
purchases.groupby(['Brand','PurchasePrice'])[['Quantity','Dollars']].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,4640,51921.60
5215,9.41,4923,46325.43
5255,9.35,6215,58110.25


In [12]:
vendor_invoice['PONumber'].nunique()

NameError: name 'vendor_invoice' is not defined

In [ ]:
sales.groupby('Brand')[['SalesDollars','SalesPrice','SalesQuantity']].sum()

* The purchase table contains actucal purchase date, including the data of purchase, product (branch) purchased by vendors, the amount paid(in dollars) and the quantity purchased.
* The purchase price column is dervied from the purchase_prices table, which provides proudct-wise actual and purchase price. The combination of vendor and brand is unique in this table.
* The Vendor_invoice table aggregates data from the purchasese tbale, summarizing quantity and dollar amounts, along with an additional column for frieght. this table maintaines uniqueness based on vendor and PO number.
* The sales table captures actual sales transactions, detailing the brand purchased by vendors, the quantity sold the selling price, and the revenur earned.

  -----------------------------------------------------------------------------------------------------------------------------------------------------
  As the data that we need for analysis is distributed in different tables, we need to create a summary table containing:
  * Purchase transactions made by vendors
  * Sales transaction data
  * Actual product price from vendors
  * Frieght costs each vendor

In [ ]:
sales.columns

In [ ]:
frieght_summary = pd.read_sql("""SELECT
                                        VendorNumber,
                                        SUM(Freight) as Frieght 
                                 FROM vendor_invoice 
                                 GROUP BY VendorNumber""",conn)
frieght_summary

In [ ]:
pd.read_sql(""" 
    SELECT 
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.PurchasePrice,
            pp.Volume,
            pp.price AS ActualPrice,
            SUM(p.Quantity) AS totalPurchaseQuantity,
            SUM(p.Dollars) AS totalPurchaseDollars
    FROM    purchases p 
    JOIN    purchase_prices pp ON p.Brand = pp.Brand
    WHERE   p.purchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand
    ORDER BY totalPurchaseDollars
""", conn)

In [ ]:
pd.read_sql(""" 
    SELECT 
            VendorNO,
            Brand,
            SUM(SalesDollars) AS totalSalesDollars,
            SUM(SalesPrice)    AS totalsalesPrice,
            SUM(ExciseTax)     AS totalExciesTax
    FROM    sales
    GROUP BY VendorNo, Brand
    ORDER BY totalsalesPrice
""",conn )

In [4]:
#start = time.time
vendor_sales_summary = pd.read_sql(""" 
    WITH FreightSummary AS (
                                SELECT
                                        VendorNumber,
                                        SUM(Freight) AS FreightCost
                                FROM    Vendor_invoice
                                GROUP BY VendorNumber
    ),
    PurchaseSummary AS (
                                SELECT 
                                        p.VendorNumber,
                                        p.VendorName,
                                        p.Brand,
                                        p.Description,
                                        p.PurchasePrice,
                                        pp.PurchasePrice,
                                        pp.Price AS ActualPrice,
                                        pp.Volume,
                                        SUM(p.Quantity) AS TotalPurchaseQuantity,
                                        SUM(p.Dollars)  AS TotalPurchaseDollars
                                FROM    purchases p
                                JOIN    purchase_prices pp 
                                        ON p.Brand = pp.Brand
                                WHERE   p.purchasePrice > 0
                                GROUP BY p.vendorNumber, p.vendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
    ),
    SalesSummary AS (
                                SELECT 
                                        VendorNo,
                                        Brand,
                                        SUM(SalesQuantity) AS TotalSalesQuantity,
                                        SUM(SalesDollars)  AS TotalSalesDollars,
                                        SUM(SalesPrice)    AS TotalSalesPrice,
                                        SUM(ExciseTax)     AS TotalExciseTax
                                FROM    sales
                                GROUP BY VendorNO, Brand                
    )
      SELECT 
              ps.VendorNumber,
              ps.VendorName,
              ps.Brand,
              ps.Description,
              ps.PurchasePrice,
              ps.ActualPrice,
              ps.Volume,
              ps.TotalPurchaseQuantity,
              ps.TotalPurchaseDollars,
              ss.TotalSalesQuantity,
              ss.TotalSalesDollars,
              ss.TotalSalesPrice,
              ss.TotalExciseTax,
              fs.FreightCost
      FROM    PurchaseSummary ps
      LEFT JOIN SalesSummary as ss 
              ON ps.VendorNumber = ss.VendorNo
      AND     ps.Brand == ss.Brand
      LEFT JOIN FreightSummary fs
              ON ps.VendorNumber = fs.VendorNumber
      ORDER BY ps.TotalPurchaseDollars DESC
                                      
""",conn
    )

#end = time.time
#total_time = (end - start)/60 


In [6]:
vendor_sales_summary['TotalPurchaseQuantity'].sum()

np.int64(33582362)

This Query generates a vendor-wise sales and purchase summary, which is valubale for: 

**Performance Optimization:**
* The query involves heavey joined and aggregation on learge datasets like sales and purchases.
* Storing the pre-aggregated results avoids repeated expensive computations
* Helps in analyzing sales, purchasese, and pricing for different vendors and brands.
* Future Benefits of storing this data for faster Dashboarding & Reporting
* Instead of running expensive queries each time, dashbaord can fetch quickly from vendor_sales_summary 

In [18]:
vendor_sales_summary.dtypes

VendorNumber               int64
VendorName                object
Brand                      int64
Description               object
PurchasePrice            float64
ActualPrice              float64
Volume                    object
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesQuantity       float64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalExciseTax           float64
FreightCost              float64
dtype: object

In [19]:
vendor_sales_summary['VendorName'].unique()

array(['BROWN-FORMAN CORP          ', 'MARTIGNETTI COMPANIES',
       'PERNOD RICARD USA          ', 'DIAGEO NORTH AMERICA INC   ',
       'BACARDI USA INC            ', 'JIM BEAM BRANDS COMPANY    ',
       'MAJESTIC FINE WINES        ', 'ULTRA BEVERAGE COMPANY LLP ',
       'STOLI GROUP,(USA) LLC      ', 'PROXIMO SPIRITS INC.       ',
       'MOET HENNESSY USA INC      ', 'CAMPARI AMERICA            ',
       'SAZERAC CO INC             ', 'CONSTELLATION BRANDS INC   ',
       'M S WALKER INC             ', 'SAZERAC NORTH AMERICA INC. ',
       'PALM BAY INTERNATIONAL INC ', 'REMY COINTREAU USA INC     ',
       'SIDNEY FRANK IMPORTING CO  ', 'E & J GALLO WINERY         ',
       'WILLIAM GRANT & SONS INC   ', 'HEAVEN HILL DISTILLERIES   ',
       'DISARONNO INTERNATIONAL LLC', 'EDRINGTON AMERICAS         ',
       'CASTLE BRANDS CORP.        ', 'SOUTHERN WINE & SPIRITS NE ',
       'STE MICHELLE WINE ESTATES  ', 'TRINCHERO FAMILY ESTATES   ',
       'MHW LTD                    ', 'W

In [20]:
vendor_sales_summary['Description'].unique()

array(['Jack Daniels No 7 Black', "Tito's Handmade Vodka",
       'Absolut 80 Proof', ..., 'Crown Royal Apple',
       'Concannon Glen Ellen Wh Zin', 'The Club Strawbry Margarita'],
      dtype=object)

In [21]:
vendor_sales_summary['Volume'] = vendor_sales_summary['Volume'].astype('float64')

In [22]:
vendor_sales_summary.fillna(0, inplace=True)

In [23]:
vendor_sales_summary['VendorName'].str.strip()

0               BROWN-FORMAN CORP
1           MARTIGNETTI COMPANIES
2               PERNOD RICARD USA
3        DIAGEO NORTH AMERICA INC
4        DIAGEO NORTH AMERICA INC
                   ...           
10687              WINE GROUP INC
10688              SAZERAC CO INC
10689    HEAVEN HILL DISTILLERIES
10690    DIAGEO NORTH AMERICA INC
10691        PROXIMO SPIRITS INC.
Name: VendorName, Length: 10692, dtype: object

In [24]:
vendor_sales_summary['GrossProfit'] = vendor_sales_summary['TotalSalesDollars'] - vendor_sales_summary['TotalPurchaseDollars']

In [25]:
vendor_sales_summary['ProfitMargin'] = (vendor_sales_summary['GrossProfit'] / vendor_sales_summary['TotalSalesDollars'])*100

In [26]:
vendor_sales_summary['StockTurnover'] = vendor_sales_summary['TotalSalesQuantity'] / vendor_sales_summary['TotalPurchaseQuantity']

In [27]:
vendor_sales_summary['SalesPurchaseRatio'] = vendor_sales_summary['TotalSalesDollars'] / vendor_sales_summary['TotalPurchaseDollars']

In [ ]:
# Adding the summary table to database. when we are done with analysis we have to add it back to database to analysis it with power BI. 
# this is the way in real companys uses
# here we are creating empty table
cursor = conn.execute(
    """
    CREATE TABLE vendor_sales_summary (
                    VendorNumber               INT,
                    VendorName                VARCHAR(100),
                    Brand                      INT,
                    Description               VARCHAR(100),
                    PurchasePrice            DECIMAL(10,2),
                    ActualPrice              DECIMAL(10,2),
                    Volume                   DECIMAL(10,2),
                    TotalPurchaseQuantity      INT,
                    TotalPurchaseDollars     DECIMAL(15,2),
                    TotalSalesQuantity       DECIMAL(15,2),
                    TotalSalesDollars        DECIMAL(15,2),
                    TotalSalesPrice          DECIMAL(15,2),
                    TotalExciseTax           DECIMAL(15,2),
                    FreightCost              DECIMAL(15,2),
                    GrossProfit              DECIMAL(15,2),
                    ProfitMargin             DECIMAL(15,2),
                    StockTurnover            DECIMAL(15,2),
                    SalesPurchaseRatio       DECIMAL(15,2),
                    PRIMARY KEY (VendorNUmber, Brand)
        );
    """
)

In [29]:
# checking if table is created 
pd.read_sql_query('SELECT * FROM Vendor_sales_summary LIMIT 5',conn)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,142049.0,5101919.51,672819.31,260999.20,68601.68,1290667.91,25.297693,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,160247.0,4819073.49,561512.37,294438.66,144929.24,1015032.27,21.062810,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,187140.0,4538120.60,461140.15,343854.07,123780.22,1119816.92,24.675786,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,200412.0,4475972.88,420050.01,368242.80,257032.07,1214774.94,27.139908,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,135838.0,4223107.62,545778.28,249587.83,257032.07,1199901.61,28.412764,0.983556,1.396897


In [30]:
# Inserting the data into the table
vendor_sales_summary.to_sql('vendor_sales_summary', conn, if_exists = 'replace', index = False)

10692

In [34]:
import pandas as pd
import sqlite3
import logging 
from ingestion_db import ingest_db

logging.basicConfig(
    filename = "logs/get_vendor_summary.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"
)

def create_vendor_summary(conn):
    """ This function will merge the different table to get the over all vendor summary and adding new columns in the resultant data """
    vendor_sales_summary = pd.read_sql_query(""" 
            WITH FreightSummary AS (
                                        SELECT
                                                VendorNumber,
                                                SUM(Freight) AS FreightCost
                                        FROM    Vendor_invoice
                                        GROUP BY VendorNumber
            ),
            PurchaseSummary AS (
                                        SELECT 
                                                p.VendorNumber,
                                                p.VendorName,
                                                p.Brand,
                                                p.Description,
                                                p.PurchasePrice,
                                                pp.PurchasePrice,
                                                pp.Price AS ActualPrice,
                                                pp.Volume,
                                                SUM(p.Quantity) AS TotalPurchaseQuantity,
                                                SUM(p.Dollars)  AS TotalPurchaseDollars
                                        FROM    purchases p
                                        JOIN    purchase_prices pp 
                                                ON p.Brand = pp.Brand
                                        WHERE   p.purchasePrice > 0
                                        GROUP BY p.vendorNumber, p.vendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
            ),
            SalesSummary AS (
                                        SELECT 
                                                VendorNo,
                                                Brand,
                                                SUM(SalesQuantity) AS TotalSalesQuantity,
                                                SUM(SalesDollars)  AS TotalSalesDollars,
                                                SUM(SalesPrice)    AS TotalSalesPrice,
                                                SUM(ExciseTax)     AS TotalExciseTax
                                        FROM    sales
                                        GROUP BY VendorNO, Brand                
            )
              SELECT 
                      ps.VendorNumber,
                      ps.VendorName,
                      ps.Brand,
                      ps.Description,
                      ps.PurchasePrice,
                      ps.ActualPrice,
                      ps.Volume,
                      ps.TotalPurchaseQuantity,
                      ps.TotalPurchaseDollars,
                      ss.TotalSalesQuantity,
                      ss.TotalSalesDollars,
                      ss.TotalSalesPrice,
                      ss.TotalExciseTax,
                      fs.FreightCost
              FROM    PurchaseSummary ps
              LEFT JOIN SalesSummary as ss 
                      ON ps.VendorNumber = ss.VendorNo
              AND     ps.Brand == ss.Brand
              LEFT JOIN FreightSummary fs
                      ON ps.VendorNumber = fs.VendorNumber
              ORDER BY ps.TotalPurchaseDollars DESC
                                              
        """,conn
            )
    return vendor_sales_summary

def clean_data(df):
    ''' This function will clean the data '''
    # changing data type to float
    df['Volume'] = df['Volume'].astype('float')

    # Filling missing values with 0
    df.fillna(0, inplace = True)

    # Removing spaces from categorical columns
    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()

    # Creating new columns for better analysis
    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars'])*100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
    df['SalesPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']

    return df 

if __name__ == '__main__':
    # Creating database connection
    conn = sqlite3.connect('inventory.db')

    logging.info('Creating Vendor Summary Table....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())

    logging.info('Cleaning Data....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head)

    logging.info('Ingesting data...')
    ingest_db(clean_df,'vendor_sales_summary',conn)
    logging.info('completed')


In [14]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging 
import time

logging.basicConfig(
    filename = "logs/ingestion_db.log",
    level = logging.DEBUG,
    format = ("%(asctime)s - %(levelname)s - %(message)s"),
    filemode = "a"
)

engine = create_engine('sqlite:///inventory.db')

def ingest_db(df, table_name, engine):
    ''' This function will ingest dataframe into database table '''
    df.to_sql(table_name, con = engine, if_exists = 'replace', index = False)

def load_raw_data():
    ''' This function will load CSVs as dataframe and ingest inot db '''
    start = time.time()
    for file in os.listdir('data/Raw_data'):        
        if file.endswith('.csv'):
            file_path = os.path.join('data', file)
            df = pd.read_csv(file_path)
            logging.info(f"Ingesting {file} in db")
            # file[:-4] -- here i m doing the index slicing removing the .csv from the file name
            ingest_db(df, file[:-4], engine)
    end = time.time()
    # when we subtract end - start it gives in seconds hence we divide it by 60 to convert it to mins
    total_time = (end - start)/60    
    logging.info('----Ingestion Complet----')
    
    logging.info(f'time taken to complet the ingestion: {total_time}')

if __name__ == '__main__':
    load_raw_data()


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\abhis\\Desktop\\Vendor-Performance-Analysis-Python-SQL-Power-BI\\notebooks\\logs\\ingestion_db'